# Student Placement Prediction
## INT234 – Predictive Analytics

In [ ]:
# 1. Import libraries

from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display, Image
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_validate
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, log_loss, confusion_matrix
ROOT = Path.cwd()
if not (ROOT / "Data").exists(): ROOT = ROOT.parent
DATA_PATH = ROOT / "Data" / "student_placement_train.csv"
REPORTS = ROOT / "reports"
FIGURES = REPORTS / "figures"
sns.set_theme(style="whitegrid")

In [ ]:
# 2. Load and inspect the dataset

df = pd.read_csv(DATA_PATH)
print("Dataset shape:", df.shape)
display(df.head())
display(pd.DataFrame({"dtype":df.dtypes.astype(str),"missing":df.isna().sum()}))
print("Duplicate rows:", df.duplicated().sum())

In [ ]:
# 3. Target distribution and data quality

display(df["placed"].value_counts().rename(index={0:"Not Placed",1:"Placed"}).to_frame("Students"))
display((df["placed"].value_counts(normalize=True)*100).round(2).rename(index={0:"Not Placed",1:"Placed"}).to_frame("Percentage"))
print("Total missing cells:", int(df.isna().sum().sum()))
display(df.isna().sum().loc[lambda x:x>0].to_frame("Missing values"))

In [ ]:
# 4. Exploratory Data Analysis

fig, axes = plt.subplots(1,2,figsize=(13,4))
sns.countplot(data=df,x="placed",color="steelblue",ax=axes[0])
axes[0].set_xticks([0,1],["Not Placed","Placed"]); axes[0].set_title("Placement distribution")
miss=df.isna().sum().sort_values(ascending=False); miss=miss[miss>0].sort_values()
miss.plot(kind="barh",ax=axes[1]); axes[1].set_title("Missing values by feature")
plt.tight_layout(); plt.show()

In [ ]:
if "cgpa" in df:
    sns.histplot(data=df,x="cgpa",hue="placed",kde=True,stat="density",common_norm=False)
    plt.title("CGPA distribution by placement"); plt.show()
if "preferred_domain" in df:
    (df.groupby("preferred_domain")["placed"].mean().sort_values(ascending=False)
       .plot(kind="bar",figsize=(8,4),title="Placement rate by preferred domain"))
    plt.ylabel("Placement rate"); plt.tight_layout(); plt.show()
if "internships_count" in df:
    df.groupby("internships_count")["placed"].mean().plot(marker="o",figsize=(7,4),title="Placement rate by internship count")
    plt.ylabel("Placement rate"); plt.tight_layout(); plt.show()

In [ ]:
numeric_for_corr=df.select_dtypes(include=np.number).drop(columns=["student_id"],errors="ignore")
plt.figure(figsize=(12,9)); sns.heatmap(numeric_for_corr.corr(),cmap="coolwarm",center=0)
plt.title("Correlation heatmap"); plt.tight_layout(); plt.show()
display(numeric_for_corr.corr()["placed"].drop("placed").sort_values(key=abs,ascending=False).head(10))

In [ ]:
# 5. Preprocessing and feature preparation

if not set(df["placed"].dropna().unique()).issubset({0,1}):
    raise ValueError("Target must contain only 0 and 1.")
y=df["placed"].astype(int)
X=df.drop(columns=["placed","student_id"],errors="ignore")
X_train,X_test,y_train,y_test=train_test_split(X,y,test_size=0.20,stratify=y,random_state=42)
numeric=X_train.select_dtypes(include=np.number).columns.tolist()
categorical=X_train.select_dtypes(exclude=np.number).columns.tolist()
numeric_pipeline=Pipeline([("imputer",SimpleImputer(strategy="median")),("scaler",StandardScaler())])
categorical_pipeline=Pipeline([("imputer",SimpleImputer(strategy="most_frequent")),("encoder",OneHotEncoder(handle_unknown="ignore"))])
preprocessor=ColumnTransformer([("numeric",numeric_pipeline,numeric),("categorical",categorical_pipeline,categorical)])
print("Train:",X_train.shape,"Test:",X_test.shape)
print("Numeric features:",len(numeric),"Categorical features:",len(categorical))

In [ ]:
# 6. Classification models and 5-fold cross-validation

classifiers={
 "DummyClassifier":DummyClassifier(strategy="most_frequent"),
 "Logistic Regression":LogisticRegression(class_weight="balanced",max_iter=2000,random_state=42),
 "Decision Tree":DecisionTreeClassifier(class_weight="balanced",max_depth=6,random_state=42),
 "Random Forest":RandomForestClassifier(n_estimators=200,max_depth=10,class_weight="balanced",random_state=42,n_jobs=-1)
}
cv=StratifiedKFold(n_splits=5,shuffle=True,random_state=42)
scoring={"accuracy":"accuracy","precision":"precision","recall":"recall","f1":"f1","roc_auc":"roc_auc","neg_log_loss":"neg_log_loss"}
cv_results=[]
for name,clf in classifiers.items():
    pipe=Pipeline([("preprocessor",preprocessor),("classifier",clf)])
    scores=cross_validate(pipe,X_train,y_train,cv=cv,scoring=scoring,n_jobs=-1)
    cv_results.append({"Model":name,"Accuracy":scores["test_accuracy"].mean(),
      "Precision":scores["test_precision"].mean(),"Recall":scores["test_recall"].mean(),
      "F1-score":scores["test_f1"].mean(),"ROC-AUC":scores["test_roc_auc"].mean(),
      "Log Loss":-scores["test_neg_log_loss"].mean()})
cv_df=pd.DataFrame(cv_results).sort_values("ROC-AUC",ascending=False)
display(cv_df.round(4))

In [ ]:
# 7. Held-out test evaluation

test_rows=[]; fitted={}
for name,clf in classifiers.items():
    pipe=Pipeline([("preprocessor",preprocessor),("classifier",clf)])
    pipe.fit(X_train,y_train); fitted[name]=pipe
    pred=pipe.predict(X_test); proba=pipe.predict_proba(X_test)
    test_rows.append({"Model":name,"Accuracy":accuracy_score(y_test,pred),
      "Precision":precision_score(y_test,pred,zero_division=0),
      "Recall":recall_score(y_test,pred,zero_division=0),
      "F1-score":f1_score(y_test,pred,zero_division=0),
      "ROC-AUC":roc_auc_score(y_test,proba[:,1]),"Log Loss":log_loss(y_test,proba,labels=[0,1])})
test_df=pd.DataFrame(test_rows)
display(test_df.round(4))

In [ ]:
# 8. Confusion matrices and ROC curves

from sklearn.metrics import ConfusionMatrixDisplay, roc_curve
fig,axes=plt.subplots(2,2,figsize=(11,9))
for ax,(name,pipe) in zip(axes.ravel(),fitted.items()):
    ConfusionMatrixDisplay(confusion_matrix(y_test,pipe.predict(X_test),labels=[0,1]),
        display_labels=["Not Placed","Placed"]).plot(ax=ax,cmap="Blues",values_format="d",colorbar=False)
    ax.set_title(name)
plt.tight_layout(); plt.show()
plt.figure(figsize=(7,5))
for name,pipe in fitted.items():
    prob=pipe.predict_proba(X_test)[:,1]; fpr,tpr,_=roc_curve(y_test,prob)
    plt.plot(fpr,tpr,label=f"{name} (AUC={roc_auc_score(y_test,prob):.3f})")
plt.plot([0,1],[0,1],"--",label="Chance"); plt.xlabel("False Positive Rate"); plt.ylabel("True Positive Rate")
plt.title("ROC curves"); plt.legend(); plt.show()

In [ ]:
# 9. Feature/metric interpretation and conclusion

print("Placement class balance:")
display((y.value_counts(normalize=True)*100).round(2).rename(index={0:"Not Placed",1:"Placed"}).to_frame("Percent"))
print("Highest CV ROC-AUC model:",cv_df.iloc[0]["Model"])
print("Interpretation: accuracy alone is insufficient because the target is imbalanced.")
print("ROC-AUC around 0.50 means the models have limited ranking discrimination on this dataset.")
print("The findings describe this Kaggle dataset and do not establish real-world placement causation.")